In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import xarray as xr
from datetime import datetime


In [2]:
# Information about the data locations
rootDir = "/cyfast/adelhass/models/nemo/out/eORCA1_ELIC_6_ref3/"

In [3]:
# Define regions with latitude and longitude bounds
# "The five sectors of Raphael and Hobbs (2014): East Antarctica (71–163◦ E), Ross/Amundsen (163–250◦ E), Amundsen/Bellingshausen (250–293◦ E), Weddell (293–346◦ E), and King Hakon VII (346–71◦ E)."
regions = {
    "Southern Ocean": {"lat_min": -90, "lat_max": 0, "lon_min": -180, "lon_max": 180},
    "East Antarctica": {"lat_min": -90, "lat_max": 0, "lon_min": 71, "lon_max": 163},
    "Ross/Amundsen": {"lat_min": -90, "lat_max": 0, "lon_min": 163, "lon_max": -110},
    "Amundsen/Bellingshausen": {"lat_min": -90, "lat_max": 0, "lon_min": -110, "lon_max": -67},
    "Weddell Sea": {"lat_min": -90, "lat_max": 0, "lon_min": -67, "lon_max": -14},
    "King Hakon VII Sea": {"lat_min": -90, "lat_max": 0, "lon_min": -14, "lon_max": 71},
    "Arctic Ocean": {"lat_min": 66, "lat_max": 90, "lon_min": -180, "lon_max": 180},
}

In [4]:
# Information about experiments. This is a dictionary where keys are experiment names
# and values are lists containing a description, start date, end date, and number of ensemble members.
expinfo = {
    "as01": ["SIC data assimilation experiment", datetime(1970, 1, 1), datetime(2023, 12, 31), 25],
}

# define period of analysis
start_date = datetime(1980, 1, 1)
end_date = datetime(2023, 12, 31)

# Define period for climatology
datebclim = datetime(1981, 1, 1)
dateeclim = datetime(2010, 12, 31)


In [5]:
# Loading the data.
exp="as01"

dateb=expinfo[exp][1]
yearb=dateb.year
monb=dateb.month
dayb=dateb.day

datee=expinfo[exp][2]
yeare=datee.year
mone=datee.month
daye=datee.day

nmemb = 25# expinfo[exp][3] # Use 3 for coding, then move to expinfo[exp][3]


# Read one file per member
# We are creating a list of xarray datasets, one for each member, which we will later concatenate along a new "member" dimension.
datasets = []

for m in range(1, nmemb + 1):
    # Example file for member 1: as01_m25_1m_1970-01-01_2023-12-31_sivolu.nc

    fileIn = rootDir + "/" + exp + "/" + f"{exp}_m{m:02d}_1m_{yearb:04d}-{monb:02d}-{dayb:02d}_{yeare:04d}-{mone:02d}-{daye:02d}_sivolu.nc"

    # Test file existence
    if not os.path.isfile(fileIn):
        raise FileNotFoundError(f"File not found: {fileIn}")
    else:
        print(f"File found: {fileIn}")

    ds = xr.open_dataset(fileIn)

    # Subset to period of analysis
    ds = ds.sel(time_counter=slice(start_date, end_date))
    

    # Load the sea ice volu- sivolu  variable
    sivolu = ds["sivolu"]

    # Load the grid cell area variable
    areacello = ds["cell_area"]

    # Load the latitude and longitude variables
    lat = ds["nav_lat"]
    lon = ds["nav_lon"]
    # Center longitudes if necessary (-180 to 180)
    lon = ((lon + 180) % 360) - 180

    # Multiply sea ice volume by grid cell area to get total sea ice volume per grid 
    # cell and divide by 1e12 to convert from m3 to thousands of km3
    sivolu_total = sivolu * areacello / 1e12  # in thousands of km3

    # Save the total sea ice volume for this member in a new variable
    ds["sivolu_total"] = sivolu_total       
    datasets.append(ds)

# Check that sivolu total is distinct for two different members
print(datasets[0]["sivolu_total"].isel(time_counter=0, y=315, x=95).values)
print(datasets[1]["sivolu_total"].isel(time_counter=0, y=315, x=95).values)



File found: /cyfast/adelhass/models/nemo/out/eORCA1_ELIC_6_ref3//as01/as01_m01_1m_1970-01-01_2023-12-31_sivolu.nc
File found: /cyfast/adelhass/models/nemo/out/eORCA1_ELIC_6_ref3//as01/as01_m02_1m_1970-01-01_2023-12-31_sivolu.nc
File found: /cyfast/adelhass/models/nemo/out/eORCA1_ELIC_6_ref3//as01/as01_m03_1m_1970-01-01_2023-12-31_sivolu.nc
File found: /cyfast/adelhass/models/nemo/out/eORCA1_ELIC_6_ref3//as01/as01_m04_1m_1970-01-01_2023-12-31_sivolu.nc
File found: /cyfast/adelhass/models/nemo/out/eORCA1_ELIC_6_ref3//as01/as01_m05_1m_1970-01-01_2023-12-31_sivolu.nc
File found: /cyfast/adelhass/models/nemo/out/eORCA1_ELIC_6_ref3//as01/as01_m06_1m_1970-01-01_2023-12-31_sivolu.nc
File found: /cyfast/adelhass/models/nemo/out/eORCA1_ELIC_6_ref3//as01/as01_m07_1m_1970-01-01_2023-12-31_sivolu.nc
File found: /cyfast/adelhass/models/nemo/out/eORCA1_ELIC_6_ref3//as01/as01_m08_1m_1970-01-01_2023-12-31_sivolu.nc
File found: /cyfast/adelhass/models/nemo/out/eORCA1_ELIC_6_ref3//as01/as01_m09_1m_1970-0

In [6]:
# Loop over regions and calculate total sea ice volume
region_volumes = {}
for region_name, bounds in regions.items():
    print(f"Processing region: {region_name}")
    lat_min = bounds["lat_min"]
    lat_max = bounds["lat_max"]
    lon_min = bounds["lon_min"]
    lon_max = bounds["lon_max"]

    # Create a mask for the region
    lat_mask = (lat >= lat_min) & (lat <= lat_max)
    if lon_min < lon_max: # This is to deal with regions crossing the dateline
        lon_mask = (lon >= lon_min) & (lon <= lon_max)
    else:
        lon_mask = (lon >= lon_min) | (lon <= lon_max)

    region_mask = lat_mask & lon_mask


    # Apply the mask to the sea ice volume data, for each member
    for ds in datasets:
        sivolu_region = ds["sivolu_total"].where(region_mask, drop=True)

        # Sum over latitude and longitude to get total volume for the region
        total_volume = sivolu_region.sum(dim=["y", "x"])
    
        print(total_volume.values[100])



        # Store the result for this member
        if region_name not in region_volumes:
            region_volumes[region_name] = []
        region_volumes[region_name].append(total_volume)


        


Processing region: Southern Ocean
6.3888173
6.2095103
6.1073284
6.275016
6.000723
5.758604
6.076048
6.0853953
6.2207737
6.413581
6.469985
6.276001
6.0633006
6.055545
6.4343987
6.6004972
6.130621
5.7125883
6.282779
5.8917103
6.2932453
6.049602
6.0627365
5.6998854
6.385366
Processing region: East Antarctica
0.7919158
0.75317454
0.85682976
0.7580941
0.7948582
0.6481823
0.8033989
0.7517441
0.79382217
0.8368378
0.88912064
0.83273655
0.71069735
0.74503833
0.8497603
0.80313015
0.8224682
0.6826966
0.76302254
0.7820433
0.7757807
0.74693143
0.773434
0.69250244
0.8433112
Processing region: Ross/Amundsen
1.8467351
1.6591213
1.7640209
1.788292
1.724494
1.4919116
1.4779447
1.830359
1.7602293
1.6625531
1.8599887
1.7896467
1.8163848
1.7796338
1.8152893
2.089285
1.715632
1.5775307
1.7173204
1.4988096
1.6916364
1.6470273
1.4965004
1.7410425
1.8995074
Processing region: Amundsen/Bellingshausen
0.69515663
0.61891496
0.57645583
0.5463096
0.5387815
0.5253104
0.56794405
0.6353333
0.5856898
0.6462302
0.543707

In [7]:
# Plot the raw time series for each region, for each member, and save the figures

for region_name, total_volumes in region_volumes.items():
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    for i, total_volume in enumerate(total_volumes):
        total_volume.plot(ax=ax, label=f"Member {i+1}")
    ax.set_title(f"Total Sea Ice Volume in {region_name} ({exp})")
    ax.set_ylabel("Sea Ice Volume (thousands km³)")
    ax.set_xlabel("Time")
    ax.legend()     
    # Create a safe filename by replacing spaces, slashes with underscores
    safe_region_name = region_name.replace(" ", "_").replace("/", "_")
    fig.savefig(f"./figs/check_sivolu_{safe_region_name}_{exp}.png") 


# print(region_volumes["Southern Ocean"][1])

In [8]:
# Produce one figure per region, with upper plot showing the mean seasonal cycle
# over the twelve months of the year (labeled J, F, ..., D), for all members,
# and lower plot showing the anomalies time series (raw signal 
# minus seasonal cycle over the analysis period for each member),
# together with smoothed version (12-month running mean) of these anomalies of the ensemble mean

# Make sure the figures display grids, the first one has the lower limit at 0, 
# and the second one as zero at the center of the y-axis.

for region_name, total_volumes in region_volumes.items():
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), dpi = 300)

    # Calculate the mean seasonal cycle for each member
    seasonal_cycles = []
    for total_volume in total_volumes:
        # Group by month and calculate the mean for each month, for the period
        # for seasonal cycle calculation (datebclim to dateeclim)
        # Note that datebclim and and dateeclim are datetime
        # objects and therefore do not support > and < operators with the time_counter variable, which is a numpy datetime64 object.

        clim_mask = (total_volume["time_counter"] >= np.datetime64(datebclim)) & (total_volume["time_counter"] <= np.datetime64(dateeclim)) 

        total_volume_clim = total_volume.sel(time_counter=clim_mask)
        seasonal_cycle = total_volume_clim.groupby("time_counter.month").mean(dim="time_counter")
        seasonal_cycles.append(seasonal_cycle)


    # Plot the mean seasonal cycle for each member
    for i, seasonal_cycle in enumerate(seasonal_cycles):
        seasonal_cycle.plot(ax=ax1, label=f"Member {i+1}", linewidth=0.5)
    ax1.set_title(f"Mean Seasonal Cycle of Total Sea Ice Volume in {region_name} ({exp})")
    ax1.set_ylabel("Sea Ice Volume (thousands km³)")
    # Replace the ticks on the x-axis with month names
    ax1.set_xticks(np.arange(1, 13))
    ax1.set_xlim(1, 13)
    ax1.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
    ax1.set_title("Mean seasonal cycle of total sea ice volume in " + region_name +
                   " period " + datebclim.strftime("%Y-%m-%d") + " to " + dateeclim.strftime("%Y-%m-%d"))
    ax1.grid()

    # Calculate anomalies (raw signal minus seasonal cycle) for each member
    anomalies = []
    for total_volume, seasonal_cycle in zip(total_volumes, seasonal_cycles):
        # Expand the seasonal cycle to match the time dimension of total_volume
        expanded_seasonal_cycle = seasonal_cycle.sel(month=total_volume["time_counter.month"])
        anomaly = total_volume - expanded_seasonal_cycle
        anomalies.append(anomaly)

    # Plot the anomalies time series for each member
    for i, anomaly in enumerate(anomalies):
        anomaly.plot(ax=ax2, label=f"Member {i+1}", alpha=0.5, linewidth=0.5)
    
    # Calculate and plot the smoothed version (12-month running mean) of the ensemble mean anomaly
    ensemble_mean_anomaly = xr.concat(anomalies, dim="member").mean(dim="member")
    smoothed_ensemble_mean_anomaly = ensemble_mean_anomaly.rolling(time_counter=12, center=True).mean()
    smoothed_ensemble_mean_anomaly.plot(ax=ax2, label="Smoothed Ensemble Mean", color="black", linewidth=2)

    ax2.set_title(f"Anomalies of Total Sea Ice Volume in {region_name} ({exp})")
    ax2.set_ylabel("Anomaly (thousands km³)")
    ax2.set_xlabel("Time")
    ax2.grid()

    # Create a safe filename by replacing spaces, slashes with underscores
    safe_region_name = region_name.replace(" ", "_").replace("/", "_")
    fig.savefig(f"./figs/sivolu_seasonal_anomalies_{safe_region_name}_{exp}.png")